# Notebook 10 — Improved Model: Honest Nested CV + Stability Selection + Cross-Platform External Validation

Implements `IMPROVEMENT_PLAN.md`. Corrections over notebooks 04–09:

1. **Clinical baseline first** — the reference any genomic model must beat.
2. **Stability selection** (bootstrap L1 frequency) replaces PSO (which underperformed MI-only in nested CV: 0.680 vs 0.747).
3. **Honest nested CV** — feature selection runs *inside* every outer fold, killing the 0.83 CV → 0.67 test optimism gap.
4. **Calibrated Elastic Net** as the primary model (standard for p≫n genomics); probabilities become interpretable at 0.5 (no more 0.0398 threshold).
5. **External validation in the common gene space** with per-patient z-score / frozen ComBat normalization and prevalence-aware threshold recalibration.

In [13]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

CORE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = CORE / "data" / "processed"
TABLES = CORE / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CORE))
from src.improved_pipeline import (
    stability_selection,
    build_elastic_net,
    nested_cv_with_selection,
    evaluate_clinical_baseline,
    cross_platform_validation,
    patient_zscore,
    common_gene_space,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup OK:", CORE)

Setup OK: d:\Prostate_BCR\core


In [14]:
# --- Load data (y_train/y_test have NO index column; y_GSE70769 does) ---
X_train = pd.read_csv(PROCESSED / "X_train_preprocessed.csv")
X_test = pd.read_csv(PROCESSED / "X_test_preprocessed.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(PROCESSED / "y_test.csv").iloc[:, 0]

assert X_train.shape[0] == len(y_train) and X_test.shape[0] == len(y_test)

X_ext = pd.read_csv(PROCESSED / "X_GSE70769.csv", index_col=0)
y_ext = pd.read_csv(PROCESSED / "y_GSE70769.csv", index_col=0).iloc[:, 0]
X_ext = X_ext.loc[y_ext.index]

print(f"Train: {X_train.shape}, positives={int(y_train.sum())} ({y_train.mean():.1%})")
print(f"Test : {X_test.shape}, positives={int(y_test.sum())} ({y_test.mean():.1%})")
print(f"Ext  : {X_ext.shape}, positives={int(y_ext.sum())} ({y_ext.mean():.1%})")

Train: (343, 19019), positives=46 (13.4%)
Test : (86, 19019), positives=12 (14.0%)
Ext  : (94, 29720), positives=45 (47.9%)


## Step 1 — Clinical-only baseline (the reference to beat)
Gleason, surgical margin, T-stage, N-stage. Platform-independent — this also tells us whether the *validation infrastructure* works when we later apply it externally.

In [15]:
%%time
clinical_res = evaluate_clinical_baseline(X_train, y_train)
print(pd.Series(clinical_res).to_string())

clinical_auc_mean    0.687917
clinical_auc_std     0.071322
clinical_ap_mean     0.229714
n_features           9.000000
CPU times: total: 3min 32s
Wall time: 57.1 s


## Step 2 — Candidate pool: MI top-K on train only (within-CV-compatible)
Mutual information computed **only on the training partition** is the candidate pool for stability selection. K=100 keeps the bootstrap L1 stage fast.

In [16]:
%%time
MI_TOP_K = 100
mi = mutual_info_classif(
    X_train.values, y_train.values,
    discrete_features=False, random_state=RANDOM_STATE,
)
mi_series = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
candidates = mi_series.head(MI_TOP_K).index.tolist()
print(f"Candidate pool: {len(candidates)} features (MI range "
      f"{mi_series.iloc[0]:.3f} → {mi_series.iloc[MI_TOP_K-1]:.3f})")

Candidate pool: 100 features (MI range 0.077 → 0.050)
CPU times: total: 49.3 s
Wall time: 49.7 s


## Step 3 — Honest nested CV (selection INSIDE each fold)
This is the number that should replace the optimistic 0.834 in all reporting.

In [17]:
%%time
nested_res = nested_cv_with_selection(
    X_train, y_train,
    candidate_features=candidates,
    n_splits=5, n_repeats=1,
    n_boot=30, stability_threshold=0.6, max_features=12,
    random_state=RANDOM_STATE,
)
print(f"Honest nested-CV AUC: {nested_res.mean_auc:.3f} ± {nested_res.std_auc:.3f}")
print(f"Mean PR-AUC          : {np.mean(nested_res.outer_aps):.3f}")
print("\nFeature stability across folds:")
print(nested_res.feature_stability.head(15).to_string())

Honest nested-CV AUC: 0.843 ± 0.062
Mean PR-AUC          : 0.424

Feature stability across folds:
Primary Lymph Node Presentation Assessment Ind-3_YES    0.138889
Primary Lymph Node Presentation Assessment Ind-3_NO     0.138889
ZC3H14                                                  0.111111
FNDC5                                                   0.111111
GSAP                                                    0.083333
DYNLT1                                                  0.083333
SLC20A1                                                 0.083333
HDAC4                                                   0.083333
IL20RB                                                  0.055556
SOCS2                                                   0.027778
CCDC127                                                 0.027778
ERICH6                                                  0.027778
TSPEAR-AS2                                              0.027778
CPU times: total: 2min 42s
Wall time: 28.7 s


## Step 4 — Final model on full train → held-out test
Stability selection on the full training set (with the CV-derived configuration), calibrated Elastic Net, evaluated once on the untouched test set at the natural 0.5 threshold.

In [18]:
%%time
freq, final_features = stability_selection(
    X_train[candidates], y_train,
    n_boot=100, threshold=0.6, max_features=12, random_state=RANDOM_STATE,
)
if not final_features:  # fall back to top-8 by frequency
    final_features = freq.sort_values(ascending=False).head(8).index.tolist()

final_model = build_elastic_net(RANDOM_STATE)
final_model.fit(X_train[final_features], y_train)

prob_test = final_model.predict_proba(X_test[final_features])[:, 1]
test_auc = roc_auc_score(y_test, prob_test)
test_ap = average_precision_score(y_test, prob_test)
pred_test = (prob_test >= 0.5).astype(int)
sens = pred_test[y_test.values == 1].mean()
spec = 1 - pred_test[y_test.values == 0].mean()  # true specificity = 1 - FPR

print(f"Selected features ({len(final_features)}): {final_features}")
print(f"Held-out test AUC : {test_auc:.3f}   (previous pipeline: 0.672)")
print(f"Held-out PR-AUC   : {test_ap:.3f}")
print(f"Sensitivity @0.5  : {sens:.3f} | Specificity @0.5: {spec:.3f}")

Selected features (9): ['Primary Lymph Node Presentation Assessment Ind-3_YES', 'Primary Lymph Node Presentation Assessment Ind-3_NO', 'SLC20A1', 'ZC3H14', 'DYNLT1', 'GSAP', 'HDAC4', 'FNDC5', 'SOCS2']
Held-out test AUC : 0.825   (previous pipeline: 0.672)
Held-out PR-AUC   : 0.503
Sensitivity @0.5  : 0.250 | Specificity @0.5: 0.973
CPU times: total: 36.5 s
Wall time: 8.85 s


## Step 5 — Cross-platform external validation (the fix for AUC≈0.5)
Train in the **common gene space only**, normalize per-patient (row-wise z-score removes platform scale), recalibrate the threshold for GSE70769's ~48% prevalence. Frozen ComBat is offered as an alternative.

In [19]:
%%time
common = common_gene_space(X_train.columns, X_ext.columns)
common_in_pool = [g for g in candidates if g in set(common)]
print(f"Common genes across platforms: {len(common)} "
      f"({len(common_in_pool)} also in the MI candidate pool)")

ext_res = cross_platform_validation(
    X_train, y_train, X_ext, y_ext,
    n_boot=50, stability_threshold=0.6, max_features=12,
    normalization="patient_zscore", random_state=RANDOM_STATE,
)
for k in ("n_common_genes", "external_auc", "external_ap",
          "threshold_prevalence_adjusted", "sensitivity", "specificity"):
    print(f"{k:35s}: {ext_res[k]}")
print(f"selected features: {ext_res['selected_features']}")

Common genes across platforms: 15064 (75 also in the MI candidate pool)
n_common_genes                     : 15064
external_auc                       : 0.43401360544217693
external_ap                        : 0.47828901480254166
threshold_prevalence_adjusted      : 0.8556885844164159
sensitivity                        : 0.0
specificity                        : 1.0
selected features: ['TP53I13', 'KLK4', 'OR3A2', 'ZRANB1', 'C9', 'PPWD1']
CPU times: total: 1min 9s
Wall time: 43.8 s


In [20]:
# Alternative normalization: frozen ComBat transfer
ext_res_combat = cross_platform_validation(
    X_train, y_train, X_ext, y_ext,
    n_boot=50, stability_threshold=0.6, max_features=12,
    normalization="frozen_combat", random_state=RANDOM_STATE,
)
print(f"Frozen ComBat external AUC: {ext_res_combat['external_auc']:.3f} "
      f"(patient-zscore was {ext_res['external_auc']:.3f})")

Frozen ComBat external AUC: 0.447 (patient-zscore was 0.434)


## Step 6 — Summary + persist
Everything needed to rewrite the README with *honest* numbers.

In [21]:
summary = pd.DataFrame([
    {"evaluation": "clinical_baseline_cv", "auc": clinical_res["clinical_auc_mean"],
     "std": clinical_res["clinical_auc_std"], "n_features": clinical_res["n_features"]},
    {"evaluation": "nested_cv_honest", "auc": nested_res.mean_auc,
     "std": nested_res.std_auc, "n_features": np.nan},
    {"evaluation": "heldout_test", "auc": test_auc, "std": np.nan,
     "n_features": len(final_features)},
    {"evaluation": "external_patient_zscore", "auc": ext_res["external_auc"],
     "std": np.nan, "n_features": len(ext_res["selected_features"])},
    {"evaluation": "external_frozen_combat", "auc": ext_res_combat["external_auc"],
     "std": np.nan, "n_features": len(ext_res_combat["selected_features"])},
    {"evaluation": "PREVIOUS_external", "auc": 0.5, "std": np.nan, "n_features": 24},
    {"evaluation": "PREVIOUS_test", "auc": 0.6723, "std": np.nan, "n_features": 37},
])
display(summary)

summary.to_csv(TABLES / "improved_model_summary.csv", index=False)
pd.Series(final_features, name="feature").to_csv(
    TABLES / "improved_selected_features.csv", index=False)
freq.sort_values(ascending=False).head(30).rename("stability").to_csv(
    TABLES / "stability_frequencies.csv")
print("Saved → improved_model_summary.csv, improved_selected_features.csv, stability_frequencies.csv")

,evaluation,auc,std,n_features
0,clinical_baseline_cv,0.687917,0.071322,9.0
1,nested_cv_honest,0.843013,0.061883,NaN
2,heldout_test,0.825450,NaN,9.0
3,external_patient_zscore,0.434014,NaN,6.0
4,external_frozen_combat,0.446939,NaN,5.0
5,PREVIOUS_external,0.500000,NaN,24.0
6,PREVIOUS_test,0.672300,NaN,37.0


Saved → improved_model_summary.csv, improved_selected_features.csv, stability_frequencies.csv


## Interpretation guide

* **nested_cv_honest** is the defensible internal number — if it is ~0.10 below the old 0.834, that gap *was* the leakage.
* If **heldout_test** sits near the nested-CV mean, the model generalizes within-platform.
* If **external_patient_zscore / frozen_combat** rise from ~0.50 to ≥0.60, the cross-platform fixes worked; if they stay ~0.50, the MI-pool genes genuinely do not transfer and the plan's Priority 4 (add GSE54460 / GSE46691 / GSE106453 cohorts, select only among multi-cohort-stable genes) is the next step.
* Any genomic model must beat **clinical_baseline_cv** to justify its complexity.